<span style="font-size: 22px; font-weight: bold;">The Evolution of Gaming</span>

Connecting to MySQL and Preparing the Dataset

This section connects to the MySQL database **dagevolve** and retrieves all rows from the
`gamesales` table. After loading the data into a pandas DataFrame, the script performs
initial cleaning on the `North America_Sales` column to ensure it is numeric and ready
for analysis.

Steps Performed

- Establish a connection to the MySQL server using `mysql.connector`.
- Execute a SQL query to retrieve all records from the `gamesales` table.
- Convert the query results into a pandas DataFrame for easier manipulation.
- Clean the `North America_Sales` column by:
  - Replacing invalid values such as `"", "N/A", "NA", None"` with `0`
  - Converting the column to `float`, filling missing values, and finally converting to `int`
- Display the first few rows of the cleaned dataset using `df.head()`.


Cleaning the sales columns ensures that all values are numeric, which is required for
accurate visualizations, aggregations, and statistical analysis. Without this step,
plots and calculations could fail or produce misleading results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import mysql.connector
import seaborn as sns

In [ ]:


# Connecting to MySQL
conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Catalyst@312",
    database="dagevolve"
)

cursor = conn.cursor()

# Ran query manually to inspect data
cursor.execute("SELECT * FROM gamesales")
rows = cursor.fetchall()


columns = [col[0] for col in cursor.description]
df = pd.DataFrame(rows, columns=columns)

# Cleaned and converted North America_Sales
df["North America_Sales"] = (
    df["North America_Sales"]
    .replace(["", "N/A", "NA", None], 0)
    .astype(float)
    .fillna(0)
    .astype(int)
)

df.head()

In [ ]:
query = """
SELECT 
    CAST(
        NULLIF(REPLACE(North_America_Sales, ',', ''), '') 
        AS UNSIGNED
    ) AS North_America_Sales,
    *
FROM gamesales;
"""#    Execute the query and load results into a DataFrame

In [ ]:
cursor.execute("SELECT * FROM GameSales;")# 
df = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df.head()

In [ ]:
cursor.execute("DESCRIBE GameSales;")
df = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])#  Inspect the structure of the GameSales table
df.head()

In [ ]:
import pandas as pd

cursor.execute("SELECT * FROM gamesales ORDER BY Company ASC;")# to get all the data from the GameSales table ordered by Genre in descending order
df = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])#
df.head()

In [ ]:
query = """
    SELECT Genre, Global_Sales
    FROM gamesales
"""
cursor.execute(query)
rows = cursor.fetchall()
cols = [col[0] for col in cursor.description]
genre_sales = pd.DataFrame(rows, columns=cols)


In [ ]:
import pandas as pd

df_vgsales = pd.DataFrame(rows, columns=cols)
df_vgsales.head()


In [ ]:
import matplotlib.pyplot as plt


Genre = df_vgsales.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#959f99'] * len(Genre)

max_index = Genre.idxmax()
highlight_position = Genre.index.get_loc(max_index)
colors[highlight_position] = '#17a9bf'

plt.bar(Genre.index, Genre.values, color=colors, width=0.8)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.xlabel("Genre")
plt.ylabel("Global Sales (Millions)")
plt.title("Global Highest Grossing Genre")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()




In [ ]:
query = """
    SELECT Company, Global_Sales
    FROM gamesales
"""
cursor.execute(query)
rows = cursor.fetchall()
cols = [col[0] for col in cursor.description]

import pandas as pd
df_vgsales = pd.DataFrame(rows, columns=cols)
df.head()

In [ ]:
import matplotlib.pyplot as plt

Company_sales = (
    df_vgsales.groupby("Company")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(12, 7))

plt.barh(
    Company_sales["Company"],
    Company_sales["Global_Sales"],
    color="#08770e"
)

plt.title("Top 10 Highest Grossing Game Companies", fontsize=16)
plt.xlabel("Global Sales (Millions)", fontsize=12)
plt.ylabel("Company", fontsize=12)

plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
query = """
    SELECT 
        Console,
        Global_Sales,
        
        Japan_Sales,
        Europe_Sales
    FROM gamesales
"""
cursor.execute(query)
rows = cursor.fetchall()
cols = [col[0] for col in cursor.description]

df_vgsales = pd.DataFrame(rows, columns=cols)
df.head()


In [ ]:
# Create top 10 for each region
stack_df = (
    na_sales[["Console", "North America_Sales"]]
    .merge(jp_sales[["Console", "Japan_Sales"]], on="Console")
    .merge(eu_sales[["Console", "Europe_Sales"]], on="Console")
    .merge(global_sales[["Console", "Global_Sales"]], on="Console")
)

na_sales = (
    df.groupby("Console")["North America_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

eu_sales = (
    df.groupby("Console")["Europe_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

jp_sales = (
    df.groupby("Console")["Japan_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)



# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- 1. Global: Bar Chart ---
axes[0, 0].bar(global_sales["Console"], global_sales["Global_Sales"], color="#0878F7")
axes[0, 0].set_title("Top 10 Consoles – Global Sales")
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 1].yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)

# --- 2. North America: Scatter Plot ---
axes[0, 1].scatter(na_sales["Console"], na_sales["North America_Sales"], color="#08061E", s=120)
axes[0, 1].set_title("Top 10 Consoles – North America")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)

# --- 3. Europe: Horizontal Bar Chart ---
axes[1, 1].barh( eu_sales["Console"], eu_sales["Europe_Sales"], color="#ED600E")
axes[1, 1].set_title("Top 10 Consoles - Europe")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].xaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)

# --- 4. Japan: Line Plot ---
axes[1, 0].plot(jp_sales["Console"], jp_sales["Japan_Sales"], marker="o", color="#470738")
axes[1, 0].set_title("Top 10 Consoles | Japan")
axes[1, 0].tick_params(axis="x", rotation=45)

axes[1, 0].yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)




plt.tight_layout()
plt.show()



In [ ]:
# Merge the three region DataFrames on Console
stack_df = (
    na_sales[["Console", "North America_Sales"]]
    .merge(eu_sales[["Console", "Europe_Sales"]], on="Console")
    .merge(jp_sales[["Console", "Japan_Sales"]], on="Console")
)

# Create stacked bar chart
plt.figure(figsize=(14, 8))

plt.bar(
    stack_df["Console"],
    stack_df["North America_Sales"],
    label="North America",
    color="#1E0ECD"
)

plt.bar(
    stack_df["Console"],
    stack_df["Europe_Sales"],
    bottom=stack_df["North America_Sales"],
    label="Europe",
    color="#ED9B0E"
)

plt.bar(
    stack_df["Console"],
    stack_df["Japan_Sales"],
    bottom=stack_df["North America_Sales"] + stack_df["Europe_Sales"],
    label="Japan",
    color="#470738"
)

plt.gca().yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)
plt.title("Console Sales by Region")
plt.xlabel("Console")
plt.ylabel("Sales (Millions)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
query = """
SELECT 
    'North America' AS Region,
    SUM(`North America_Sales`) AS Total_Sales
FROM gamesales

UNION ALL

SELECT 
    'Europe' AS Region,
    SUM(Europe_Sales) AS Total_Sales
FROM gamesales

UNION ALL

SELECT 
    'Japan' AS Region,
    SUM(Japan_Sales) AS Total_Sales
FROM gamesales;
"""
cursor.execute(query)
rows = cursor.fetchall()
cols = [col[0] for col in cursor.description]
df_region_sales = pd.DataFrame(rows, columns=cols)

plt.pie(
    df_region_sales["Total_Sales"],
    labels=df_region_sales["Region"],
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("viridis", len(df_region_sales)),
    explode = [0, 0, 0.14]   # NA, EU, JP
    #explode = [
    #0.15 if v == df_region_sales["Total_Sales"].max() else 0
    #for v in df_region_sales["Total_Sales"]#  to explode the slice with the highest sales
    #]
    #explode = [0.05] * len(df_region_sales)   # explode all slices slightly

)



plt.title("Total Video Game Sales by Region (Millions)")
plt.tight_layout()
plt.show()
